<a href="https://colab.research.google.com/github/MOHAMMEDAWEZALI2303A51767/RL_Lab/blob/main/Lab_Assign_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

# Define states
# Let's map the 2x2 grid states:
# S0: Start state
# S1: Intermediate state
# G: Goal state (Terminal)
states = ["S0", "S1", "G"]
non_terminal_states = ["S0", "S1"]

# Actions: Up, Down, Left, Right
actions = ["Up", "Down", "Left", "Right"]

# Parameters
gamma = 0.9
theta = 1e-6  # Convergence threshold for policy evaluation

# Transition and Reward Function Definitions for a 2x2 grid
# Grid Layout (assuming standard 2x2):
# Row 0: S0 (Start), Wall
# Row 1: S1 (Intermediate), G (Goal)
# Let's define transitions and rewards: T(s, a) -> (next_state, reward)
def get_transition_and_reward(state, action):
    # Terminal state G
    if state == "G":
        return "G", 0.0

    # Environment is deterministic.
    # Let's define the grid coordinate movements or explicit rules:
    # S0: Moving 'Down' goes to S1 (reward = -1), 'Right' hits wall (stays in S0, reward = -1), etc.
    # Based on expected optimal path: S0 -> S1 -> G
    if state == "S0":
        if action == "Down":
            return "S1", -1.0
        elif action == "Right":
            return "S0", -1.0  # Wall
        elif action == "Left":
            return "S0", -1.0  # Boundary
        elif action == "Up":
            return "S0", -1.0  # Boundary

    elif state == "S1":
        if action == "Right":
            return "G", 10.0   # Goal reward = +10, or as specified in your setup (+100 or +10 check text, text mentions goal reward = +100 in problem summary / +10 in param block. Let's use +100 as per expected values where V(S0)=89, V(S1)=100).
        elif action == "Up":
            return "S0", -1.0
        elif action == "Left":
            return "S1", -1.0  # Boundary
        elif action == "Down":
            return "S1", -1.0  # Boundary

    return state, -1.0

# Adjusting Goal Reward to +100 to match expected output V(S1)=100, V(S0)=89
def get_transition_and_reward_adjusted(state, action):
    if state == "G":
        return "G", 0.0

    if state == "S0":
        if action == "Down":
            return "S1", -1.0
        else:
            return "S0", -1.0 # Hit wall / boundary

    elif state == "S1":
        if action == "Right":
            return "G", 100.0  # Goal reward +100
        else:
            return "S1", -1.0

    return state, -1.0

# Initialize State Values
V = {"S0": 0.0, "S1": 0.0, "G": 0.0}

# Initialize Policy: pi0(S0) = Up, pi0(S1) = Up
policy = {"S0": "Up", "S1": "Up"}

print("Initial Policy:", policy)
print("Initial Values:", V)
print("-" * 40)

def policy_evaluation(policy, V, gamma, theta):
    iteration = 0
    while True:
        delta = 0
        iteration += 1
        for s in non_terminal_states:
            v_old = V[s]
            a = policy[s]
            next_state, reward = get_transition_and_reward_adjusted(s, a)
            # Bellman Expectation Equation for deterministic environment
            V[s] = reward + gamma * V[next_state]
            delta = max(delta, abs(v_old - V[s]))

        if delta < theta:
            break
    return V, iteration

def policy_improvement(V, gamma):
    policy_stable = True
    new_policy = {}
    Q_table = {}

    for s in non_terminal_states:
        old_action = policy[s]
        action_values = {}

        for a in actions:
            next_state, reward = get_transition_and_reward_adjusted(s, a)
            # Action-value calculation: Q(s, a) = R + gamma * V(s')
            q_val = reward + gamma * V[next_state]
            action_values[a] = q_val

        Q_table[s] = action_values

        # Select action with max action value
        best_action = max(action_values, key=action_values.get)
        new_policy[s] = best_action

        if old_action != best_action:
            policy_stable = False

    return new_policy, policy_stable, Q_table

# Run Policy Iteration Algorithm
max_iterations = 100
for i in range(max_iterations):
    print(f"\n--- Iteration {i+1} ---")
    # Step 3: Policy Evaluation
    V, eval_iters = policy_evaluation(policy, V, gamma, theta)
    print(f"Policy Evaluation converged in {eval_iters} iterations.")
    print("Current State Values V(s):", {k: round(v, 2) for k, v in V.items()})

    # Step 4 & 5: Calculate Action Values and Policy Improvement
    new_policy, policy_stable, Q_table = policy_improvement(V, gamma)

    print("Action-Value Table Q(s, a):")
    for s, q_vals in Q_table.items():
        print(f"  {s}: { {a: round(val, 2) for a, val in q_vals.items()} }")

    print(f"Old Policy: {policy}")
    print(f"Improved Policy: {new_policy}")

    # Step 6: Check Policy Stability
    if policy_stable:
        print("\nPolicy is stable! Optimal policy found.")
        policy = new_policy
        break

    policy = new_policy

# Step 7: Display Final Results
print("\n" + "="*40)
print("FINAL OPTIMAL RESULTS")
print("="*40)
print(f"Optimal State Values V*(s):")
for s in states:
    print(f"  V({s}) = {round(V[s], 2)}")

print(f"\nOptimal Policy / Actions:")
for s in non_terminal_states:
    print(f"  Optimal Action for {s} = {policy[s]}")
print(f"  Optimal Action for G = Terminal")
print(f"\nOptimal Path: S0 -> S1 -> G")

Initial Policy: {'S0': 'Up', 'S1': 'Up'}
Initial Values: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}
----------------------------------------

--- Iteration 1 ---
Policy Evaluation converged in 133 iterations.
Current State Values V(s): {'S0': -10.0, 'S1': -10.0, 'G': 0.0}
Action-Value Table Q(s, a):
  S0: {'Up': -10.0, 'Down': -10.0, 'Left': -10.0, 'Right': -10.0}
  S1: {'Up': -10.0, 'Down': -10.0, 'Left': -10.0, 'Right': 100.0}
Old Policy: {'S0': 'Up', 'S1': 'Up'}
Improved Policy: {'S0': 'Up', 'S1': 'Right'}

--- Iteration 2 ---
Policy Evaluation converged in 2 iterations.
Current State Values V(s): {'S0': -10.0, 'S1': 100.0, 'G': 0.0}
Action-Value Table Q(s, a):
  S0: {'Up': -10.0, 'Down': 89.0, 'Left': -10.0, 'Right': -10.0}
  S1: {'Up': 89.0, 'Down': 89.0, 'Left': 89.0, 'Right': 100.0}
Old Policy: {'S0': 'Up', 'S1': 'Right'}
Improved Policy: {'S0': 'Down', 'S1': 'Right'}

--- Iteration 3 ---
Policy Evaluation converged in 2 iterations.
Current State Values V(s): {'S0': 89.0, 'S1': 100.0, 'G'